# 실제 CCTV Identity / Track-heldout 평가

이 노트북은 MOV 촬영 조건과 평가 gate를 확인하고, 검수된 track manifest가 준비되면 동일한 코드로 모델별 Rank-1·Recall@K·mAP·오탐률을 계산한다. 현재는 identity label이 없으므로 프로젝트 점수를 계산하지 않고 차단 상태를 표시한다.

기준 문서: `docs/CCTV_TRACK_HELDOUT_PROTOCOL_V1_1.md`

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'experiments').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

def read_json(relative_path: str) -> dict:
    return json.loads((PROJECT_ROOT / relative_path).read_text(encoding='utf-8'))

profile = read_json('experiments/results/cctv_local_video_profile.json')
status = read_json('experiments/results/cctv_identity_evaluation_status.json')
reconciliation = read_json('experiments/results/cctv_contract_reconciliation.json')
print(profile['status'])
print(status['status'])
print(reconciliation['projectIdentityGate'])

In [ ]:
video_rows = []
for video in profile['videos']:
    video_rows.append({
        'file': video['fileName'],
        'resolution': f"{video['width']}x{video['height']}",
        'fps': video['fps'],
        'duration_s': video['durationSeconds'],
        'orientation': video['orientation'],
    })
video_rows

In [ ]:
from qwen_backend.cctv_identity_evaluation import (
    TrackReference,
    TrackRetrievalPrediction,
    evaluate_identity_predictions,
)

references = (
    TrackReference(caseId='demo', videoId='demo', cameraId='cam', conditionGroupId='landscape', trackId='target-01', split='test', targetRole='target', identityGroupId='person-01', frameCount=12),
    TrackReference(caseId='demo', videoId='demo', cameraId='cam', conditionGroupId='landscape', trackId='distractor-01', split='test', targetRole='distractor', identityGroupId='person-02', frameCount=12),
)
predictions = (
    TrackRetrievalPrediction(queryTrackId='target-01', candidates=({'identityGroupId': 'person-01', 'score': 0.9}, {'identityGroupId': 'person-02', 'score': 0.2}), decision='match'),
    TrackRetrievalPrediction(queryTrackId='distractor-01', candidates=({'identityGroupId': 'person-01', 'score': 0.8},), decision='reject'),
)
demo_report = evaluate_identity_predictions(references, predictions, model_name='protocol-demo')
demo_report.model_dump(by_alias=True)

In [ ]:
assert len(profile['videos']) == 3
assert status['identityLabelsAvailable'] is False
assert status['trackHeldoutMetricsEligible'] is False
assert reconciliation['projectIdentityGate']['eligibleModels'] == 0
assert demo_report.rank1 == 1.0
print('PASS: profile, fail-closed status, and track-level calculator are consistent')